# Databricks Data Engineering - Study Notebook (Platinum)
*Built from 41 pages of handwritten notes (bronze -> silver -> gold), de-duplicated, corrected and enriched with 2025-2026 changes. Written 2026-11-08.*

**How to use**
- Import into a Databricks workspace (*Workspace -> Import -> File*, choose this `.ipynb`).
- Attach to **serverless** compute or a cluster on **Databricks Runtime 17.x / 18.x** with **Unity Catalog** enabled.
- Cells are small, independent samples. Run chapter 0 first (creates a scratch schema and a volume).
- Legend: **[NOTES]** = from the handwritten notes, **[UPDATE]** = newer information not in the notes (verify against the current docs, product names move fast), **[FIX]** = correction of a statement in the notes.

**Chapters**
0. Setup
1. Platform architecture & compute
2. Spark fundamentals (execution model, DAG, shuffle)
3. DataFrame API (incl. Pandas vs PySpark)
4. SQL on Databricks (nested data, JSON/VARIANT, arrays, windows, scripting)
5. Delta Lake
6. Ingestion: COPY INTO, Auto Loader, Structured Streaming, Kafka
7. Lakeflow Spark Declarative Pipelines (ex-Delta Live Tables)
8. Unity Catalog & governance
9. Performance tuning
10. Data modelling: medallion & Data Vault
11. dbt
12. DevOps: Git folders, Asset Bundles, CI/CD
13. Orchestration, migration & ingestion tools (ADF, Fivetran, Lakeflow Connect)
14. Data products & data contracts
15. What changed since the notes were written (cheat sheet)

## 0. Setup
Widgets let you change the catalog/schema without editing code. On new workspaces a catalog called `workspace` usually exists; on others use `main` or a catalog you own.

In [ ]:
dbutils.widgets.text("catalog", "main", "Catalog")
dbutils.widgets.text("schema", "study_notes", "Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.landing")
VOL = f"/Volumes/{CATALOG}/{SCHEMA}/landing"
print("Using", CATALOG, SCHEMA, VOL)
print("Spark version:", spark.version)

def safe_conf(key, value):
    """Some configs are locked on serverless; do not fail the notebook."""
    try:
        spark.conf.set(key, value); print("set", key, "=", value)
    except Exception as e:
        print("cannot set", key, "->", str(e)[:100])

## 1. Platform architecture & compute

### 1.1 Planes **[NOTES p4, U8]**
- **Control plane** - Databricks-managed backend: web app, compute orchestration, Workflows/Jobs, notebooks and code, Unity Catalog.
- **Compute (data) plane** - where data is processed: *classic* compute in **your** subscription/VNet (workspace clusters, SQL warehouses) or *serverless* compute in Databricks' account.
- Encryption: at rest (storage service encryption, optional customer-managed keys in Key Vault), in transit (TLS).

### 1.2 Cluster types **[NOTES p4]**
| | All-purpose | Job cluster |
|---|---|---|
| Created | manually (UI/CLI/API) | automatically per job run |
| Lifetime | persistent, restartable | ephemeral |
| Users | shared | one job |
| Cost | higher | lower |
| Use | interactive analysis | ETL/ML automation |

Cluster ACL permissions: `CAN ATTACH TO`, `CAN RESTART`, `CAN MANAGE`. Auto-termination: 10-10,000 minutes.

### 1.3 Spark layout on a cluster **[NOTES p1, 7, 15]**
Driver (JVM running the app, schedules tasks) + executors (JVMs on worker nodes, usually 1 per node). Each CPU core is a **slot**; one partition = one task = one slot.

### 1.4 Runtime versioning **[NOTES p4]**
GA, LTS and Beta releases; each version fixes the Spark/Python/Scala/Delta versions and has an end-of-support date.

### 1.5 Pools **[NOTES p4]**
Pools keep idle, ready VMs to cut cluster start and autoscale time. No DBUs are charged for idle pool instances, but the cloud VM cost applies.

### 1.6 [UPDATE] What is different in 2026
- **Serverless compute** (notebooks, jobs, SQL warehouses, pipelines) is the default recommendation: no cluster sizing, no pools, instant start.
- **Databricks Runtime 18** is based on **Apache Spark 4.1.0** (GA June 2026 per Databricks release notes); 17.x is on Spark 4.0.
- Access modes are now **standard** (shared) and **dedicated** (single user); "high concurrency" mode is gone.
- **DBFS root and DBFS mounts are deprecated**; use Unity Catalog **volumes** / external locations.
- Azure AD is now **Microsoft Entra ID**.
- Fabric "Default dataset" -> "default semantic model" (Fabric comparison from p3).

### 1.7 Lakehouse vs warehouse vs lake **[NOTES p15]**
| | Warehouse | Lake | Lakehouse |
|---|---|---|---|
| ACID | yes | no | yes |
| Schema on write | yes | no | yes |
| Query speed | fast | slow | fast |
| Data types | structured | all | all |
| Cost of scale | expensive | cheaper | cheaper |
| Uses | BI | ML | BI + ML |

In [ ]:
# Inspect the environment you are attached to
print("Spark:", spark.version)
try:
    print("Runtime:", spark.conf.get("spark.databricks.clusterUsageTags.sparkVersion"))
except Exception:
    print("Runtime tag not available (probably serverless)")
print("Default parallelism:", spark.sparkContext.defaultParallelism if hasattr(spark, "sparkContext") else "n/a (Spark Connect)")
display(spark.sql("SELECT current_catalog() AS catalog, current_schema() AS schema, current_user() AS user"))

## 2. Spark fundamentals

### 2.1 Execution model **[NOTES p7, 20, 21, 22]**
- **Action** (`count`, `collect`, `show`, `display`, `write`) -> triggers a **job**.
- A job is split into **stages** at every **shuffle** (wide transformation).
- A stage is split into **tasks**, one per partition.
- The **DAG** (directed acyclic graph) is the logical plan of stages/tasks: *directed* because operations run in order, *acyclic* because there are no cycles.
- **Transformations are lazy** **[FIX: the notes say "transactions" on p7]**: nothing runs until an action is called.
- *Narrow* transformations (`select`, `filter`, `withColumn`) work inside one partition and are **pipelined** into one task; *wide* ones (`groupBy`, `join`, `distinct`, `repartition`) shuffle data between executors.
- Every distributed operation maps to **Map -> Shuffle -> Reduce** (groupBy: extract key, shuffle by key, aggregate; filter: map only).
- Query path: SQL / DataFrame API -> logical plan -> **Catalyst** optimized plan -> physical plan -> RDDs -> execution. Catalyst is rule-based except the choice among physical plans (cost-based).
- **Tungsten**: binary in-memory format (UnsafeRow) and whole-stage code generation; Spark works directly on binary data without turning rows into JVM objects.
- **Apache Arrow** speeds up JVM <-> Python transfers.

### 2.2 Storage levels **[NOTES p20]**
`MEMORY_ONLY`, `MEMORY_ONLY_SER`, `MEMORY_AND_DISK`, `MEMORY_AND_DISK_SER`, `DISK_ONLY`. `cache()` = default level; `persist(level)` lets you choose. *(Not available on serverless: use Delta caching/disk cache or write intermediate tables.)*

### 2.3 Resource settings **[NOTES p20]**
`spark.dynamicAllocation.enabled` (scale executors with load), `spark.speculation` (re-launch slow tasks), `spark.locality.wait` (how long to wait for a data-local slot).

In [ ]:
from pyspark.sql import functions as F

df = spark.range(0, 1_000_000).withColumn("bucket", F.col("id") % 10).withColumn("value", F.rand(42))

# Lazy: nothing has run yet
agg = df.filter("value > 0.5").groupBy("bucket").agg(F.count("*").alias("n"), F.avg("value").alias("avg_value"))

# Look at the plan: Exchange = shuffle = stage boundary
agg.explain("formatted")

In [ ]:
# Action -> job. Open the Spark UI / query profile to see jobs, stages and tasks.
display(agg.orderBy("bucket"))
print("partitions of source df:", df.rdd.getNumPartitions() if hasattr(df, "rdd") else "n/a on serverless")

### 2.4 No-op traps **[NOTES p31]**
Operations on non-existent columns do not always fail:

In [ ]:
df2 = df.withColumnRenamed("does_not_exist", "x")   # silently does nothing
df3 = df.drop("does_not_exist")                       # silently does nothing
print(df2.columns == df.columns, df3.columns == df.columns)   # True True

## 3. DataFrame API

### 3.1 Reading / writing **[NOTES p21]**
- **Reader schema options:** explicit schema (`.schema(...)`), implicit (`header=False` -> `_c0, _c1, ...`), or `inferSchema=True` (extra pass over the data).
- **Writer modes:** `append`, `overwrite`, `error`/`errorifexists`, `ignore`.
- **Transformations:** `select, drop, withColumnRenamed, withColumn, filter/where, sort/orderBy, dropDuplicates, distinct, limit, groupBy, selectExpr`.
- **Grouped methods:** `agg, avg, count, max, min, mean, pivot, sum`.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

csv_text = "id,name,dept_id\n1,Ann,10\n2,Bob,20\n3,Cleo,10\n4,Dan,\n"
dbutils.fs.put(f"{VOL}/employees.csv", csv_text, overwrite=True)

schema = StructType([StructField("id", IntegerType()), StructField("name", StringType()), StructField("dept_id", IntegerType())])
emp_explicit = spark.read.schema(schema).option("header", True).csv(f"{VOL}/employees.csv")
emp_infer    = spark.read.option("header", True).option("inferSchema", True).csv(f"{VOL}/employees.csv")
emp_implicit = spark.read.option("header", False).csv(f"{VOL}/employees.csv")   # _c0, _c1, _c2 (header becomes data)
emp_explicit.printSchema(); emp_implicit.printSchema()
display(emp_explicit)

### 3.2 Pandas vs PySpark **[NOTES p13-14]**
| Pandas | PySpark |
|---|---|
| `pd.read_csv(path, header=0)` | `spark.read.csv(path, header=True)` |
| `df.head(n)` | `df.take(n)` / `df.show(n)` / `df.limit(n)` (returns a DataFrame) |
| `df['c']` | `df['c']`, `df.c`, `F.col('c')` |
| `df['new'] = 2*df['c']` | `df.withColumn('new', 2*F.col('c'))` |
| `df.rename(columns={'a':'b'})` | `df.withColumnRenamed('a','b')` |
| `df.drop('c', axis=1)` | `df.drop('c')` |
| `df[df.c > 1]` | `df.filter(F.col('c') > 1)` |
| `df.c.unique()` | `df.select('c').distinct()` |
| `df.sort_values('c')` | `df.orderBy('c')` |
| `pd.concat([a, b])` | `a.union(b)` (or `unionByName`) |
| `df.c.mean()` | `df.select(F.mean('c'))` |
| `a.merge(b, on='k')` | `a.join(b, 'k', 'inner')` |

**[FIX]** `df.collect` and `df.cache` need parentheses. `pandas-profiling` is now `ydata-profiling`. For pandas code at scale, the **pandas API on Spark** (`import pyspark.pandas as ps`) is an alternative.

In [ ]:
dept = spark.createDataFrame([(10, "Sales"), (20, "IT"), (30, "HR")], ["dept_id", "dept_name"])

# join types: inner, left (outer), right (outer), full, left_semi, left_anti, cross
display(emp_explicit.join(dept, "dept_id", "left"))
display(emp_explicit.join(dept, "dept_id", "left_anti"))      # employees with no matching dept (incl. NULL dept)

# nulls
display(emp_explicit.na.fill({"dept_id": 0}))
display(emp_explicit.na.drop(subset=["dept_id"]))

### 3.3 Aggregates, `collect_list` vs `collect_set`, `explode` **[NOTES p7, 22, 32]**
`collect_list` keeps duplicates, `collect_set` removes them; `explode` turns an array into rows, `collect_list` inside `agg` turns rows back into an array. `flatten` collapses nested arrays.

**[FIX]** `sumDistinct` is deprecated: use `sum_distinct`.

In [ ]:
people = spark.createDataFrame([("u1", ["hiking", "ml", "photo"]), ("u2", ["ml", "ml", "chess"])], ["user", "interests"])
exploded = people.select("user", F.explode("interests").alias("interest"))
display(exploded)
display(exploded.groupBy("user").agg(F.collect_list("interest").alias("as_list"), F.collect_set("interest").alias("as_set")))
display(spark.createDataFrame([([[1, 2], [3]],)], ["nested"]).select(F.flatten("nested").alias("flat")))

### 3.4 RDDs **[NOTES p14-15]**
RDDs (immutable, partitioned, in-memory collections) still exist but DataFrames are preferred (Catalyst + Tungsten). Use RDDs for fine control over partitioning or legacy code. Transformations (`map`, `flatMap`, `filter`) return RDDs; actions (`reduce`, `count`) return values. *RDD APIs are not available on serverless / Spark Connect.*

**UDFs:** Python UDFs run row by row (slow); prefer built-in functions, pandas UDFs (Arrow), or SQL UDFs that Catalyst can inline.

In [ ]:
from pyspark.sql.functions import udf, pandas_udf
import pandas as pd

@udf("string")
def shout(s): return None if s is None else s.upper() + "!"

@pandas_udf("string")
def shout_vec(s: pd.Series) -> pd.Series: return s.str.upper() + "!"

display(emp_explicit.select("name", shout("name").alias("udf"), shout_vec("name").alias("pandas_udf")))

## 4. SQL on Databricks
Databricks SQL follows **ANSI** semantics; `spark.sql()` lets you embed existing SQL unchanged and returns a DataFrame. After a `%sql` cell the result is available in Python as `_sqldf`. **[NOTES p14, U4]**

### 4.1 Counting, missing data and de-duplication **[NOTES p5, 30]**
- `count(*)` counts rows, `count(col)` counts non-NULL values, `count_if(cond)` counts rows where the condition is true, `count(col) FILTER (WHERE cond)` is the standard form.
- Missing data: `count_if(email IS NULL)`.
- De-duplicate: `SELECT DISTINCT *`, `ROW_NUMBER()`/`QUALIFY`, `MERGE`, or `CREATE TABLE AS SELECT DISTINCT *`.
- Pattern to collapse partially filled duplicates: `GROUP BY id` with `max(email)` etc., then check `count(*) = 1` per id.

In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE users_raw AS
SELECT * FROM VALUES
 (1,'ann@x.com','PL'), (1,NULL,'PL'), (2,'bob@x.com',NULL), (2,'bob@x.com',NULL), (3,NULL,NULL)
 AS t(id,email,country)
""")
display(spark.sql("""
SELECT count(*) AS rows_,
       count(email) AS non_null_email,
       count_if(email IS NULL) AS null_email,
       count(*) FILTER (WHERE country = 'PL') AS pl_rows
FROM users_raw
"""))
# collapse duplicates, filling gaps with max()
users = spark.sql("SELECT id, max(email) AS email, max(country) AS country FROM users_raw GROUP BY id")
display(users)
assert users.groupBy("id").count().filter("count > 1").count() == 0   # one record per id

### 4.2 Nested data, JSON and VARIANT **[NOTES p5, 6, 7, 30]**
- **struct** fields: dot syntax `address.city`, expand with `address.*`.
- **JSON strings**: colon syntax `col:field`, or `from_json(col, schema)`; schema can come from a *literal* sample via `schema_of_json('{...}')` **[FIX: it does not accept a column]**.
- Complex types: `ARRAY`, `MAP`, `STRUCT`.
- **[UPDATE] `VARIANT`** (Spark 4.0 / DBR 15.3+): semi-structured type, several times faster than JSON strings, flexible with evolving schemas: `parse_json`, `variant_get`, `col:path::type`.

In [ ]:
data = [('{"timestamp":"2026-01-01T10:00:00","details":{"code":404,"tags":["a","b"]}}',)]
raw = spark.createDataFrame(data, ["json_str"])

# 1) struct via from_json with a literal sample
sample = '{"timestamp":"2026-01-01T10:00:00","details":{"code":404,"tags":["a","b"]}}'
parsed = raw.select(F.from_json("json_str", F.schema_of_json(F.lit(sample))).alias("j"))
display(parsed.select("j.timestamp", "j.details.code", "j.details.tags"))
display(parsed.select("j.details.*"))

# 2) VARIANT
v = raw.selectExpr("parse_json(json_str) AS v")
v.createOrReplaceTempView("v_view")
display(spark.sql("SELECT v:timestamp::timestamp AS ts, v:details.code::int AS code, variant_get(v, '$.details.tags[0]', 'string') AS first_tag FROM v_view"))

### 4.3 Arrays and higher-order functions **[NOTES p6]**
`transform`, `filter`, `aggregate`/`reduce`, `exists`, `explode`, `flatten`, `collect_*`. Lambda syntax: `x -> expr`.

In [ ]:
display(spark.sql("""
SELECT arr,
       transform(arr, x -> x * 10)                AS times10,
       filter(arr, x -> x % 2 = 0)                AS evens,
       aggregate(arr, 0, (acc, x) -> acc + x)      AS total,
       exists(arr, x -> x > 4)                     AS has_gt4
FROM VALUES (array(1,2,3,4,5)) AS t(arr)
"""))

### 4.4 Regex, CASE, window functions **[NOTES p5, 6, 38]**
- `regexp_extract` (first match) / `regexp_extract_all` (all matches).
- Window functions: `PARTITION BY` splits like `GROUP BY` but keeps all rows; `ORDER BY` defines order for running totals; frames such as `ROWS BETWEEN 6 PRECEDING AND CURRENT ROW` give moving averages; `ROW_NUMBER()` numbers rows.
- **[UPDATE] `QUALIFY`** filters on window results without a subquery (Databricks SQL).

In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE fact_salary AS
SELECT * FROM VALUES (1,'F',100),(2,'M',120),(3,'F',90),(4,'M',130),(5,'F',110) AS t(emp_id,gender,salary)
""")
display(spark.sql("""
SELECT *,
  sum(salary)  OVER (PARTITION BY gender ORDER BY emp_id)                       AS rolling_total,
  avg(salary)  OVER (ORDER BY emp_id ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)  AS moving_avg_3,
  row_number() OVER (PARTITION BY gender ORDER BY salary DESC)                  AS rn
FROM fact_salary
"""))
# top earner per gender with QUALIFY
display(spark.sql("SELECT * FROM fact_salary QUALIFY row_number() OVER (PARTITION BY gender ORDER BY salary DESC) = 1"))
display(spark.sql(r"SELECT regexp_extract('order-4711-PL', '(\\d+)', 1) AS first_number, regexp_extract_all('a1b22c333', '(\\d+)', 1) AS all_numbers"))

### 4.5 Parameters, variables, UDFs, scripting **[NOTES p6, U4, U5, 30]** + **[UPDATE]**
- Parametrize with **widgets**, **parameter markers** (`:name`, `?`) and **SQL session variables** (`DECLARE VARIABLE`, DBR 14.1+).
- **SQL UDFs** (`CREATE FUNCTION ... RETURN ...`) are Catalyst-transparent when inlined; Python UDFs are opaque. Use dedicated schemas to govern team UDFs.
- Missing T-SQL functions can be wrapped in a UDF (e.g. `GETDATE()` -> `current_timestamp()`).
- **[FIX/UPDATE]** The notes say "no stored procedures, no loops". Since Spark 4.0 / DBR 16.3 **SQL scripting** (`BEGIN ... END`, `IF`, `WHILE`, `FOR`, `DECLARE`, exception handlers) exists and **SQL stored procedures** (`CREATE PROCEDURE`) arrived in Unity Catalog in DBR 17. Orchestrating notebooks with **Lakeflow Jobs** (ex-Workflows) is still the standard for pipelines.
- **[UPDATE] Pipe syntax** `|>` (Spark 4.0 / DBR 16.2+): linear query flow, no lateral-alias problems.

In [ ]:
# parameter markers
display(spark.sql("SELECT * FROM fact_salary WHERE salary > :min_salary", args={"min_salary": 100}))

# SQL UDF in Unity Catalog (T-SQL GETDATE() compatibility example)
spark.sql("CREATE OR REPLACE FUNCTION getdate() RETURNS TIMESTAMP RETURN current_timestamp()")
display(spark.sql("SELECT getdate() AS now"))

# pipe syntax
display(spark.sql("""
FROM fact_salary
|> WHERE salary >= 100
|> AGGREGATE count(*) AS n, avg(salary) AS avg_salary GROUP BY gender
|> ORDER BY gender
"""))

In [ ]:
# SQL scripting (needs DBR 16.3+ / serverless). Loops and variables:
try:
    display(spark.sql("""
    BEGIN
      DECLARE i INT DEFAULT 0;
      DECLARE total INT DEFAULT 0;
      WHILE i < 5 DO
        SET i = i + 1;
        SET total = total + i;
      END WHILE;
      SELECT total AS sum_1_to_5;
    END
    """))
except Exception as e:
    print("SQL scripting not available on this compute:", str(e)[:150])

In [ ]:
# widgets as parameters (notes p14)
dbutils.widgets.text("param_name", "param_value")
var1 = dbutils.widgets.get("param_name")
print(var1)

### 4.6 Inspecting tables **[NOTES p30]**
`DESCRIBE TABLE t` (columns/types/comments), `DESCRIBE TABLE EXTENDED t` (+ location, provider, properties), `DESCRIBE HISTORY t` (Delta log).

In [ ]:
display(spark.sql("DESCRIBE TABLE EXTENDED fact_salary"))

## 5. Delta Lake
Delta Lake is an open-source storage layer (Parquet files + transaction log) that adds relational semantics to a data lake. **[NOTES p2, U9]**

**Capabilities:** ACID transactions (**[FIX]** not "transformations" as written on U9), scalable metadata, time travel, open format, unified batch + streaming source/sink, schema enforcement and evolution, full Spark API compatibility. Managed tables are Delta by default in Unity Catalog.

**Dropping tables:** dropping a *managed* table removes metadata **and data**; dropping an *external* table removes only metadata. **[FIX p2]**

**Interoperability (UniForm) [NOTES p18]:** Delta tables can expose Iceberg (and Hudi) metadata while data stays in Parquet: `delta.universalFormat.enabledFormats = 'iceberg'`.

In [ ]:
# Create via DataFrame write (append / overwrite) or SQL
orders = spark.createDataFrame([(1, "A", 10.0), (2, "B", 20.0), (3, "A", 5.0)], ["order_id", "customer", "amount"])
orders.write.mode("overwrite").saveAsTable("orders")                 # version 0
spark.createDataFrame([(4, "C", 7.5)], ["order_id", "customer", "amount"]).write.mode("append").saveAsTable("orders")   # version 1
display(spark.sql("DESCRIBE HISTORY orders"))

### 5.1 Time travel **[NOTES p2, U6]**
`VERSION AS OF n` / `TIMESTAMP AS OF ts` in SQL; `versionAsOf` / `timestampAsOf` options in the DataFrame reader. `RESTORE TABLE` rolls back.

In [ ]:
display(spark.sql("SELECT * FROM orders VERSION AS OF 0"))
v0 = spark.read.option("versionAsOf", 0).table("orders")
print("rows v0:", v0.count(), "rows now:", spark.table("orders").count())
# spark.sql("RESTORE TABLE orders TO VERSION AS OF 0")

### 5.2 CREATE OR REPLACE vs DROP/CREATE, INSERT OVERWRITE **[NOTES p7, 18]**
`CREATE OR REPLACE TABLE` replaces content in a **single transaction** (table never disappears) and **keeps history**, so `RESTORE` still works. `DROP TABLE` + `CREATE TABLE` loses history. `INSERT OVERWRITE` also preserves history.

In [ ]:
spark.sql("CREATE OR REPLACE TABLE orders_copy AS SELECT * FROM orders")
spark.sql("CREATE OR REPLACE TABLE orders_copy AS SELECT * FROM orders WHERE amount > 6")
display(spark.sql("DESCRIBE HISTORY orders_copy").select("version", "operation"))

### 5.3 MERGE (upsert, de-duplication, CDC) **[NOTES p5, 8]**
`MERGE INTO` supports complex conditions, de-duplication and schema evolution (`WITH SCHEMA EVOLUTION`). In pipelines, `APPLY CHANGES INTO` / **AUTO CDC** is the simpler declarative form (SCD type 1 and 2, ordered by a sequence column).

In [ ]:
from delta.tables import DeltaTable

updates = spark.createDataFrame([(2, "B", 25.0), (5, "D", 99.0)], ["order_id", "customer", "amount"])
(DeltaTable.forName(spark, "orders").alias("t")
    .merge(updates.alias("s"), "t.order_id = s.order_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())
display(spark.table("orders").orderBy("order_id"))

### 5.4 Duplicates and key checks **[NOTES p5]**
Check uniqueness manually (`GROUP BY id HAVING count(*) > 1`) or enforce in pipelines with expectations. Delta supports `NOT NULL` and `CHECK` constraints (primary/foreign keys are informational only).

In [ ]:
display(spark.sql("SELECT order_id, count(*) AS c FROM orders GROUP BY order_id HAVING count(*) > 1"))
spark.sql("ALTER TABLE orders ALTER COLUMN order_id SET NOT NULL")
spark.sql("ALTER TABLE orders ADD CONSTRAINT positive_amount CHECK (amount > 0)")
try:
    spark.sql("INSERT INTO orders VALUES (9, 'Z', -1.0)")
except Exception as e:
    print("Constraint works:", str(e)[:120])

### 5.5 Layout and maintenance **[NOTES p6, U6, 14]**
- `OPTIMIZE` compacts many small files into larger ones (faster I/O).
- `VACUUM` deletes files no longer referenced and older than the retention period (default 7 days); `DRY RUN` lists candidates first.
- **Z-order** co-locates data for data skipping (high-cardinality filter columns). **V-Order** is Microsoft Fabric's write-time Parquet optimization (compression/read speed); they can be combined in Fabric.
- **[UPDATE]** For new tables prefer **liquid clustering** (`CLUSTER BY`), which replaces partitioning + Z-order, can be changed without rewriting the whole table, and `CLUSTER BY AUTO` lets Databricks choose keys. Unity Catalog managed tables can use **predictive optimization**, which runs OPTIMIZE/VACUUM for you. **Deletion vectors** speed up DELETE/UPDATE/MERGE.

In [ ]:
spark.sql("CREATE OR REPLACE TABLE events_lc (id BIGINT, ts TIMESTAMP, kind STRING) CLUSTER BY (kind, ts)")
spark.sql("INSERT INTO events_lc SELECT id, current_timestamp(), CAST(id % 5 AS STRING) FROM range(1000)")
spark.sql("OPTIMIZE events_lc")
display(spark.sql("VACUUM events_lc DRY RUN"))

### 5.6 Table properties **[NOTES p32]**
`delta.enableDeletionVectors`, `delta.columnMapping.mode` (`name`/`id`/`none`; needed for renaming/dropping columns), `delta.enableIcebergCompatV2`, `delta.universalFormat.enabledFormats`, `pipelines.reset.allowed = false` (protect a bronze table from full refresh).

In [ ]:
spark.sql("ALTER TABLE orders SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name', 'delta.minReaderVersion' = '2', 'delta.minWriterVersion' = '5')")
display(spark.sql("SHOW TBLPROPERTIES orders"))

## 6. Ingestion

### 6.1 COPY INTO **[NOTES p8]**
Idempotent, file-tracked, incremental batch load from a folder: re-running the same command does not duplicate data; good for bulk loads, migrations, validation and schema alignment (`COPY_OPTIONS ('mergeSchema'='true')`).

In [ ]:
dbutils.fs.put(f"{VOL}/copy_src/a.csv", "id,name\n1,x\n2,y\n", overwrite=True)
spark.sql("CREATE TABLE IF NOT EXISTS copy_target (id INT, name STRING)")
for i in range(2):   # the second run loads nothing: COPY INTO is idempotent
    r = spark.sql(f"""
      COPY INTO copy_target FROM '{VOL}/copy_src'
      FILEFORMAT = CSV FORMAT_OPTIONS ('header'='true','inferSchema'='true')
    """)
    display(r)

### 6.2 Auto Loader **[NOTES U12]**
`cloudFiles` source: continuous/near-real-time file ingestion with cheap file discovery (directory listing or file notification), exactly-once processing, schema inference + evolution, many formats (CSV, JSON, Parquet, Avro, text, binary...), backfill support. Sources: `abfss://`, `s3://`, `gs://`, volumes. *(`wasbs://`, `adl://` are deprecated - **use `abfss://`**.)*

In [ ]:
dbutils.fs.put(f"{VOL}/auto/file1.json", '{"id":1,"v":"a"}\n{"id":2,"v":"b"}\n', overwrite=True)

q = (spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "json")
      .option("cloudFiles.schemaLocation", f"{VOL}/_schemas/auto")
      .load(f"{VOL}/auto")
      .writeStream
      .option("checkpointLocation", f"{VOL}/_checkpoints/auto")
      .trigger(availableNow=True)
      .toTable("auto_bronze"))
q.awaitTermination()
display(spark.table("auto_bronze"))

### 6.3 Structured Streaming **[NOTES p2, U6, U9]**
`spark.readStream` / `writeStream`; `checkpointLocation` stores progress so a query restarts exactly where it stopped; output modes `append`/`update`/`complete`; triggers (`availableNow`, `processingTime`). Reading a Delta table as a stream: `skipChangeCommits` **[UPDATE: replaces `ignoreChanges`/`ignoreDeletes`]** ignores updates/deletes in the source. Stop a query with `query.stop()`. Lambda architecture (separate batch + speed layers) is largely replaced by one Delta-based streaming/batch pipeline.

In [ ]:
(spark.readStream.table("orders")
      .writeStream.option("checkpointLocation", f"{VOL}/_checkpoints/orders_copy2")
      .trigger(availableNow=True).toTable("orders_stream_copy")).awaitTermination()
display(spark.table("orders_stream_copy"))

### 6.4 Kafka as a source **[NOTES p35-37]**
*Concepts*
- **Event** = record of something that happened; **producers** publish, **consumers** subscribe, **topics** are named streams, **partitions** are the unit of parallelism (order guaranteed only inside a partition; same key -> same partition; no key -> spread), **offset** identifies a record in a partition, **consumer groups** share the load.
- **Brokers** store data; each partition has a **leader replica** (handles all reads/writes) and **followers**; **ISR** (in-sync replicas) are the candidates for promotion. Replication factor <= number of brokers. Special follower states: *slow*, *stuck*, *bootstrapping*.
- **Cluster metadata:** Zookeeper (external) was replaced by **KRaft** (Raft consensus inside Kafka; default since 3.x, **only mode since Kafka 4.0**).
- Pros: scalable, durable, low latency, replay, decoupling (fire-and-forget vs waiting for an API response). Cons: limited built-in monitoring, no wildcard topic selection, not for per-message transformation of huge payloads.
- Messaging models: queue vs publish/subscribe. Kafka Streams = library for stream processing inside your app.

The template below needs a reachable broker, so it is not executed.

In [ ]:
# TEMPLATE (do not run without a broker)
# kafka_df = (spark.readStream.format("kafka")
#     .option("kafka.bootstrap.servers", "broker1:9092")
#     .option("subscribe", "orders")
#     .option("startingOffsets", "earliest")
#     .load())
# parsed = kafka_df.select(F.col("key").cast("string"), F.from_json(F.col("value").cast("string"), "id INT, amount DOUBLE").alias("v"), "partition", "offset", "timestamp")
# parsed.writeStream.option("checkpointLocation", "/Volumes/.../_chk/kafka").toTable("bronze_orders")
print("template only")

## 7. Lakeflow Spark Declarative Pipelines (formerly Delta Live Tables)
**[NOTES p7, 8, U12]** + **[UPDATE]**

Declarative pipelines define *what* tables should contain; the engine builds the DAG, handles ordering, retries, incremental processing, expectations (data quality) and maintenance (OPTIMIZE/VACUUM).

| Notes (DLT era) | Today |
|---|---|
| `CREATE LIVE TABLE` | `CREATE MATERIALIZED VIEW` (batch, query result kept up to date) |
| "Streaming Live Table" / `CREATE STREAMING LIVE TABLE` | `CREATE STREAMING TABLE` (append-only, incremental, exactly-once) |
| `FROM STREAM source` | `FROM STREAM(source)` / `STREAM source` in streaming tables |
| `import dlt`, `@dlt.table` | `from pyspark import pipelines as dp`, `@dp.table`, `@dp.materialized_view`, `@dp.temporary_view` |
| `APPLY CHANGES INTO` | AUTO CDC (same feature, newer name; check docs) |
| Triggered vs continuous pipelines | unchanged: triggered = cheaper, higher latency; continuous = always-on, low latency |
| Development vs production mode | unchanged: dev reuses the cluster, no retries; production auto-retries and shuts compute down |
| `event_log(table(...))` | unchanged table-valued function for the pipeline event log |
| Auto-set properties `optimizeWrite`, `autoCompact`, daily VACUUM/OPTIMIZE | unchanged (plus predictive optimization) |

Streaming-table rules (notes p7): append-only source; cannot be the target of `APPLY CHANGES`, cannot define an aggregate, and source rows must not be updated/deleted (otherwise use `skipChangeCommits`).

**Pipeline code cannot run inside a plain notebook cell** - create a pipeline (Jobs & Pipelines -> ETL pipeline) and attach a file/notebook with the code below.

```python
# --- pipeline source file (Python) ---
from pyspark import pipelines as dp
from pyspark.sql import functions as F

@dp.table(comment="Raw orders ingested with Auto Loader")
def orders_bronze():
    return (spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "json")
            .load("/Volumes/main/study_notes/landing/orders"))

@dp.table(comment="Cleaned orders")
@dp.expect_or_drop("valid_amount", "amount > 0")
def orders_silver():
    return spark.readStream.table("orders_bronze").withColumn("loaded_at", F.current_timestamp())

@dp.materialized_view(comment="Revenue per customer")
def revenue_gold():
    return spark.read.table("orders_silver").groupBy("customer").agg(F.sum("amount").alias("revenue"))
```

```sql
-- --- same in SQL ---
CREATE OR REFRESH STREAMING TABLE orders_bronze
AS SELECT * FROM STREAM read_files('/Volumes/main/study_notes/landing/orders', format => 'json');

CREATE OR REFRESH STREAMING TABLE orders_silver (
  CONSTRAINT valid_amount EXPECT (amount > 0) ON VIOLATION DROP ROW
) AS SELECT *, current_timestamp() AS loaded_at FROM STREAM(orders_bronze);

CREATE OR REFRESH MATERIALIZED VIEW revenue_gold
AS SELECT customer, sum(amount) AS revenue FROM orders_silver GROUP BY customer;

-- change data capture (SCD type 2)
CREATE OR REFRESH STREAMING TABLE customers_scd2;
CREATE FLOW customers_cdc AS AUTO CDC INTO customers_scd2
FROM STREAM(customers_changes) KEYS (customer_id) SEQUENCE BY change_ts STORED AS SCD TYPE 2;
```

## 8. Unity Catalog & governance

### 8.1 Object model **[NOTES U2]**
`Metastore -> Catalog -> Schema -> (table | view | volume | model | function)`. Always address objects with **three levels**: `catalog.schema.object` (Hive metastore used two levels). Default catalogs: `hive_metastore` (legacy), `main`, `samples`, `system` (`system.access.*`, `system.billing.*`, lineage and audit tables). Default schemas: `default`, `information_schema`.

Metastore-level objects: **storage credential**, **external location**, **connection** (federation), **share / recipient / provider** (Delta Sharing). One metastore **per region**; co-locate it with the workspaces to avoid cross-region latency and keep data isolated (notes U12).

### 8.2 Setup order on Azure **[NOTES U1]**
1. Resource group; 2. **Premium** Databricks workspace; 3. ADLS Gen2 account + container; 4. *Access Connector for Azure Databricks* (managed identity) with **Storage Blob Data Contributor** on the storage account; 5. Account console -> Data -> create **metastore** (region, ADLS path `abfss://container@account.dfs.core.windows.net/`, **access connector resource ID**); 6. assign workspaces (same region). Roles: *account admin*, *metastore admin*, *workspace admin*, *marketplace admin*. **[UPDATE]** New accounts often get an automatic metastore and per-workspace catalog; **default storage** and serverless reduce manual setup.

### 8.3 Storage credentials & external locations **[NOTES U3]**
A **storage credential** = identity (access connector) to reach cloud storage; an **external location** = URL + credential. Test it (read, write, list, delete, path exists, hierarchical namespace). Then `dbutils.fs.ls('abfss://...')` works like DBFS paths - **governed by Unity Catalog**.

### 8.4 Privileges **[NOTES U3]**
`USE CATALOG`, `USE SCHEMA`, `SELECT`, `MODIFY`, `CREATE TABLE`, `READ VOLUME`... `GRANT <privilege> ON <securable> TO <principal>` / `REVOKE ... FROM ...`. Principals: users, groups, service principals. Privileges inherit downwards (catalog -> schema -> table).

### 8.5 Workspaces without Unity Catalog **[NOTES U2]**
Two-level naming, no lineage, no Delta Sharing, no storage credentials/external locations, access via SAS/keys/service principals and mounts. **Legacy** - migrate with the UC upgrade tools (`SYNC`, `CREATE TABLE ... DEEP CLONE`).

### 8.6 Lakehouse Federation **[NOTES U3, U4]**
Query external databases in place: create a **connection** (host, credentials), then a **foreign catalog**. Supported: MySQL, PostgreSQL, Redshift, Snowflake, SQL Server, Synapse, BigQuery, Databricks, Salesforce Data Cloud (more added since). Needs a UC workspace, network line of sight, DBR 13.1+ on standard/dedicated access mode, and `CREATE CONNECTION` / `CREATE FOREIGN CATALOG` privileges.

### 8.7 Delta Sharing **[NOTES U2]**
Open protocol to share live data with other organizations/platforms without copying.

### 8.8 Governance themes **[NOTES U12]**
Quality, protection & compliance, stewardship (accountable vs responsible), data management. **[UPDATE]** Lineage, tags, row filters / column masks, ABAC policies, system tables and **metric views** are the Unity Catalog tools for these.

In [ ]:
# Safe, read-only look at governance metadata
display(spark.sql("SHOW CATALOGS"))
display(spark.sql(f"SHOW SCHEMAS IN `{CATALOG}`"))
display(spark.sql(f"SHOW GRANTS ON SCHEMA `{CATALOG}`.`{SCHEMA}`"))
display(spark.sql(f"SELECT table_name, table_type FROM `{CATALOG}`.information_schema.tables WHERE table_schema = '{SCHEMA}'"))

In [ ]:
# Grant/revoke examples (commented so the notebook has no side effects on shared environments)
# spark.sql(f"GRANT USE CATALOG ON CATALOG `{CATALOG}` TO `data_readers`")
# spark.sql(f"GRANT USE SCHEMA, SELECT ON SCHEMA `{CATALOG}`.`{SCHEMA}` TO `data_readers`")
# spark.sql(f"REVOKE SELECT ON SCHEMA `{CATALOG}`.`{SCHEMA}` FROM `data_readers`")

# Volumes: governed access to non-tabular files (replaces DBFS mounts)
display(dbutils.fs.ls(VOL))

## 9. Performance tuning

### 9.1 Partitions and shuffles **[NOTES p31, 15]**
- `coalesce(n)` = **narrow** (no shuffle, can only reduce partitions, may be unbalanced); `repartition(n)` = **wide** (full shuffle, balanced, can increase).
- `spark.sql.shuffle.partitions` (default 200). Above **2000** shuffle partitions Spark switches to a more compressed map-status format - a trick when suspecting OOM during a large shuffle (notes use 2001). **[UPDATE]** With **adaptive query execution (AQE)**, on by default, partitions are coalesced and skew joins split automatically.
- Shuffle = data movement between executors; avoid it where possible (broadcast small tables, filter early, pre-aggregate).

### 9.2 Skew and salting **[NOTES p31]**
Skew = a few keys (or a flood of NULL keys) create huge partitions; some executors work while others idle. **Salting** adds a random suffix to the key on the big side and replicates the small side so the hot key is spread over many partitions. Result: lower total time, balanced use of the cluster, ability to handle bigger data.

### 9.3 Physical join strategies **[NOTES p41]**
| Strategy | When |
|---|---|
| **Broadcast hash join** | fastest; one side small; equi-join |
| **Shuffle hash join** | too big to broadcast; equi-join |
| **Sort-merge join** | both sides shuffled by key and sorted within partitions (default for big/big) |
| **Broadcast nested loop join** | non-equi joins, compares every row |
| **Cartesian product** | cross join, avoid |

### 9.4 Other **[NOTES p20, U11]**
Tune data structures (primitive types, numeric ids instead of strings, avoid deeply nested objects), serialization, memory, CPU, network; partition data sensibly; use the latest runtime; use Photon; check Spark UI / query profile and the event log. **[UPDATE]** Photon and serverless are standard; disk cache replaces the old "Delta cache" naming.

In [ ]:
safe_conf("spark.sql.shuffle.partitions", "64")
print("AQE enabled:", spark.conf.get("spark.sql.adaptive.enabled", "n/a"))

big = spark.range(0, 2_000_000).withColumn("k", F.col("id") % 100).withColumn("v", F.rand(1))
small = spark.range(0, 100).withColumnRenamed("id", "k").withColumn("label", F.lit("x"))

# default join -> inspect the physical plan
big.join(small, "k").explain()
# force broadcast
big.join(small.hint("broadcast"), "k").explain()
# SQL hints: /*+ BROADCAST(t) */, /*+ SHUFFLE_MERGE(t) */, /*+ SHUFFLE_HASH(t) */, /*+ SHUFFLE_REPLICATE_NL(t) */

In [ ]:
# Salting demo: aggregate a heavily skewed key
skewed = (spark.range(0, 1_000_000)
          .withColumn("k", F.when(F.col("id") % 10 < 9, F.lit("hot")).otherwise(F.concat(F.lit("k"), (F.col("id") % 50).cast("string"))))
          .withColumn("v", F.lit(1)))

SALT = 8
salted = skewed.withColumn("salt", (F.rand(7) * SALT).cast("int"))
stage1 = salted.groupBy("k", "salt").agg(F.sum("v").alias("partial"))     # hot key is split in 8 pieces
result = stage1.groupBy("k").agg(F.sum("partial").alias("total"))         # recombine
display(result.orderBy(F.desc("total")).limit(5))

# NULL keys in joins never match but all land in one partition: filter/handle them first
# fact.filter(F.col("k").isNotNull()).join(dim, "k").unionByName(fact.filter(F.col("k").isNull()) ...)

In [ ]:
# coalesce vs repartition: compare plans
df_c = big.coalesce(1);  df_r = big.repartition(1)
print("coalesce plan:");    df_c.explain()
print("repartition plan:"); df_r.explain()

## 10. Data modelling

### 10.1 Medallion (multi-hop) architecture **[NOTES p17]**
**Bronze** = raw data, as ingested (append-only, keep history, add metadata); **Silver** = cleansed/conformed (here: the Data Vault model); **Gold** = business-level, de-normalized, read-optimized data marts with few joins. Unity Catalog naming example: `catalog.bronze.table`.

### 10.2 Data Vault **[NOTES p16-17, 24-25]**
- **Hub** - core business concept (customer, product), holds the **business key (BK)** plus a generated **surrogate key (SK)**; append-only.
- **Link** - relationship between hubs.
- **Satellite** - descriptive attributes with history (SCD type 2: `start_date`, `end_date`, `is_current`, plus a **row hash** to detect change). A satellite never links to another satellite; a hub or link can have many satellites.
- Naming: `_hub`, `_link`, `_sat`; e.g. `core.product.product_hub (product_hub_id, product_bk)`, `core.product.product_attributes_sat (product_hub_id, hash, start_date, end_date, is_current)`; report layer `report.product.products` joins them.
- Loading order: staging/prep -> **hub first** (the id must exist in the hub before a satellite row can reference it) -> links -> satellites -> report views.

In [ ]:
# Staging data (batch 1)
stg = spark.createDataFrame([("P-1", "Chair", 49.0), ("P-2", "Table", 199.0)], ["product_bk", "name", "price"])
stg.createOrReplaceTempView("stg_product")

spark.sql("""CREATE OR REPLACE TABLE product_hub (product_hub_id BIGINT GENERATED ALWAYS AS IDENTITY, product_bk STRING, load_ts TIMESTAMP)""")
spark.sql("""CREATE OR REPLACE TABLE product_attributes_sat (product_hub_id BIGINT, hash BIGINT, name STRING, price DOUBLE,
             start_date TIMESTAMP, end_date TIMESTAMP, is_current BOOLEAN)""")

def load_vault():
    # 1) hub: append only new business keys
    spark.sql("""INSERT INTO product_hub (product_bk, load_ts)
                 SELECT DISTINCT s.product_bk, current_timestamp() FROM stg_product s
                 LEFT ANTI JOIN product_hub h ON h.product_bk = s.product_bk""")
    # 2) satellite: close changed rows, insert new versions (SCD2 via hash comparison)
    spark.sql("""
      CREATE OR REPLACE TEMP VIEW stg_hashed AS
      SELECT h.product_hub_id, xxhash64(s.name, s.price) AS hash, s.name, s.price
      FROM stg_product s JOIN product_hub h ON h.product_bk = s.product_bk""")
    spark.sql("""
      MERGE INTO product_attributes_sat t
      USING stg_hashed s
      ON t.product_hub_id = s.product_hub_id AND t.is_current = true
      WHEN MATCHED AND t.hash <> s.hash THEN UPDATE SET t.end_date = current_timestamp(), t.is_current = false""")
    spark.sql("""
      INSERT INTO product_attributes_sat
      SELECT s.product_hub_id, s.hash, s.name, s.price, current_timestamp(), NULL, true
      FROM stg_hashed s
      LEFT ANTI JOIN product_attributes_sat t ON t.product_hub_id = s.product_hub_id AND t.is_current = true AND t.hash = s.hash""")

load_vault()
# batch 2: P-2 changes price, P-3 is new
spark.createDataFrame([("P-1", "Chair", 49.0), ("P-2", "Table", 179.0), ("P-3", "Lamp", 25.0)], ["product_bk", "name", "price"]).createOrReplaceTempView("stg_product")
load_vault()
display(spark.sql("SELECT h.product_bk, s.name, s.price, s.start_date, s.end_date, s.is_current FROM product_hub h JOIN product_attributes_sat s USING (product_hub_id) ORDER BY 1, s.start_date"))

In [ ]:
# Report layer: current view of the product (gold-like)
spark.sql("""CREATE OR REPLACE VIEW products AS
SELECT h.product_bk, s.name, s.price FROM product_hub h JOIN product_attributes_sat s USING (product_hub_id) WHERE s.is_current""")
display(spark.table("products"))

## 11. dbt (data build tool) **[NOTES p23-28, U5]**

dbt transforms data **inside** the warehouse/lakehouse: you write `SELECT` statements (models) plus YAML config and Jinja; dbt builds the dependency graph and runs them. Adapters: `dbt-databricks`, `dbt-snowflake`, `dbt-bigquery`, ...

```bash
python -m venv dbt_venv && .\dbt_venv\Scripts\Activate.ps1
pip install dbt-databricks
dbt init            # scaffold; profiles.yml (~/.dbt) holds connection info
dbt debug           # test connection
```

**Project files:** `dbt_project.yml` (name, profile, defaults), `profiles.yml` (credentials), `models/` (SQL + `schema.yml`), `seeds/` (CSV), `snapshots/`, `macros/`, `tests/`, `analyses/` (compiled, not materialized), `logs/dbt.log`.

**Commands:** `dbt deps` (install packages), `parse`, `compile` (Jinja -> SQL, validates refs), `run`, `test`, `build` (run + test + seed + snapshot in DAG order), `seed`, `snapshot`, `source freshness`, `docs generate`, `docs serve`, `show`, `list`, `retry`, `clone`, `clean`, `run-operation` (call a macro), `debug`.

**Referencing:** `{{ ref('model') }}` (dependency between models), `{{ source('landing','cust') }}` (declared raw tables with alias/identifier - one place to change names, shows in lineage).

**Materializations:** `view` (default), `table`, `incremental`, `ephemeral` (inlined as CTE `__dbt__cte__<name>`, no object created), `snapshot`. Set in `dbt_project.yml` or `{{ config(materialized='table') }}`.

```sql
-- models/orders_inc.sql : incremental
{{ config(materialized='incremental', unique_key='order_id') }}
select * from {{ ref('orders_stg') }}
{% if is_incremental() %}
where updated_at >= (select max(updated_at) from {{ this }})
{% endif %}
```
`is_incremental()` is true only if the target exists, the run is not `--full-refresh`, and the model is `incremental`. `{{ this }}` = the target table.

**Snapshots (SCD type 2):** adds `dbt_scd_id`, `dbt_updated_at`, `dbt_valid_from`, `dbt_valid_to`. Strategies: `timestamp` (`updated_at`) or `check` (`check_cols=['phone','email']` or `'all'`); `invalidate_hard_deletes=True` closes deleted rows.

**Tests:** built-in `not_null`, `unique`, `accepted_values`, `relationships`; **singular** (a SQL file that must return 0 rows) and **generic** (`{% test not_empty_string(model, column_name) %} ... {% endtest %}`); a test fails when its query returns rows. **Freshness**: `warn_after`/`error_after` with `loaded_at_field`. **[UPDATE]** dbt now also has first-class **unit tests** (YAML mock inputs/expected outputs) - not the same as the macro tests from p34.

**Docs:** `description:` fields or `.md` docblocks `{% docs name %}...{% enddocs %}` referenced with `{{ doc('name') }}`.

**Jinja:** `{% %}` statements (set, if, for), `{{ }}` expressions, `{# #}` comments; plain text passes through. **Macros** are functions that return the rendered text:
```sql
{% macro to_celsius(fahrenheit_column, decimal_places=1) %}
    round(({{ fahrenheit_column }} - 32) * 5/9, {{ decimal_places }})
{% endmacro %}
-- usage: select {{ to_celsius('temp_f', 2) }} as temp_c from {{ ref('weather') }}
```
**Packages** (`packages.yml`, then `dbt deps`): `dbt-labs/dbt_utils`, `calogica/dbt_expectations` (git or hub). **One model = one database object.** **[UPDATE]** The new **dbt Fusion** engine (Rust) and dbt Cloud features exist; pin versions and check compatibility with `dbt-databricks`.

In [ ]:
# dbt's CTE idea in PySpark/SQL: reusable, readable steps
display(spark.sql("""
WITH src AS (SELECT * FROM orders),
     per_customer AS (SELECT customer, sum(amount) AS revenue FROM src GROUP BY customer)
SELECT * FROM per_customer ORDER BY revenue DESC
"""))
# A dbt 'singular test' in plain SQL: must return zero rows
display(spark.sql("SELECT * FROM orders WHERE customer IS NULL OR trim(customer) = ''"))

## 12. DevOps: Git folders, Asset Bundles, CI/CD

### 12.1 Concepts **[NOTES p5, 18, U5, U10]**
- **Continuous Integration:** merge frequently into a shared branch, run automated tests (early issue detection, faster cycle). **Continuous Delivery:** automatic promotion to staging/pre-prod, manual go-live. **Continuous Deployment:** every change passing tests goes to production automatically. DevOps -> **DataOps** (data engineering) -> **MLOps** (ML).
- Isolate environments with separate **workspaces** and/or **catalogs** (`dev`, `test`, `prod`). Keep production in a separate subscription (scale, quotas).
- Notebook sharing: share inside the workspace, export/import `.dbc`/`.ipynb`, **Git folders** (**[UPDATE]** renamed from *Repos*), Delta Sharing for data.
- Git operations in Git folders: clone, pull, push, commit, branch, diff, resolve conflicts, rebase, hard reset.
- Tools mentioned: Azure DevOps, GitHub Actions, Jenkins; repos GitHub/GitLab/Bitbucket; schedulers Airflow, Dagster, Prefect.
- Build agents (Azure DevOps): custom VM or container agents give control over OS/tools, state, VNet access. Old approach: scripts against the **REST API** (create cluster, upload notebooks, run tests, build/upload wheels).

### 12.2 Deployment tools **[NOTES p18]**
REST API (raw HTTP), **SDKs** (Python, Java, Go, R), **CLI** (includes **Databricks Asset Bundles - DABs**: infrastructure as code for jobs, pipelines, notebooks, models).

### 12.3 Asset Bundles workflow **[NOTES p19-20]** - *validate -> deploy -> run (-> destroy)*
```bash
databricks -v
databricks bundle init default-python        # templates: default-python, default-sql, dbt-sql, mlops-stacks, custom
databricks bundle validate
databricks bundle deploy -t development
databricks bundle run -t development demo01_sample_dab
databricks bundle destroy --auto-approve     # no prompt - careful!
databricks catalogs list
databricks bundle validate --output json
```
Project outline: `databricks.yml`, `resources/`, `src/`, `tests/`. Custom templates: `databricks_template_schema.json` + `databricks.yml.tmpl`.

```yaml
# databricks.yml
bundle:
  name: demo01_sample_dab

variables:
  my_user_name:
    description: Your user name
    default: someone
  my_cluster:
    description: My cluster
    type: complex
    default:
      spark_version: "17.3.x-scala2.13"      # use a current LTS
      node_type_id: "Standard_DS3_v2"
      num_workers: 2

targets:
  development:
    mode: development
    default: true
    workspace:
      host: https://adb-1234567890123456.7.azuredatabricks.net

resources:
  jobs:
    demo01_sample_dab:
      name: ${bundle.name}-${bundle.target}
      tasks:
        - task_key: main
          notebook_task:
            notebook_path: ../src/main.ipynb
          job_cluster_key: c
      job_clusters:
        - job_cluster_key: c
          new_cluster: ${var.my_cluster}
```
Default substitutions: `${bundle.name}`, `${bundle.target}`, `${workspace.file_path}`, `${workspace.root_path}`, `${resources.jobs.<name>.id}`, `${resources.models.<name>.name}`, `${resources.pipelines.<name>.name}`.

Where to run the CLI: web terminal, VS Code, or a Databricks notebook (`%sh`). **[UPDATE]** Prefer **serverless** job compute and OIDC/workload-identity federation for CI instead of long-lived tokens; Azure DevOps "datathirst" extension from the notes is legacy.

### 12.4 Calling Databricks from ADF **[NOTES U5]** - generate an access token (or use managed identity), create a linked service, use the *Notebook* activity, pass parameters via widgets.

In [ ]:
# A notebook that is parameterized by widgets can be called from ADF / Jobs / dbutils.notebook.run
dbutils.widgets.text("run_date", "2026-01-01")
run_date = dbutils.widgets.get("run_date")
# dbutils.notebook.exit(json.dumps({"status": "ok", "run_date": run_date}))
print("run_date =", run_date)

## 13. Orchestration, migration & ingestion tools

### 13.1 Migrating ADF workloads to Databricks **[NOTES p39 rerun]**
Strategies: **A. Orchestration lift-and-shift** (ADF stays, calls Databricks); **B. Hybrid** (ADF only for ingestion that Databricks cannot do, e.g. sources reachable only through a self-hosted integration runtime); **C. Full replacement** (Lakeflow Jobs + Lakeflow Connect + pipelines).

Sources that need a self-hosted IR and are **not** available in Lakeflow Connect (at the time of the notes): ODBC, SAP (workaround: SAP Business Data Cloud connector for Databricks, or JDBC to HANA), legacy databases (DB2, Netezza, Sybase), Hive, Cassandra.

ADF vocabulary: **Integration Runtime** = compute infrastructure; **Linked service** = connection string; **Dataset** = pointer to data; **Pipeline** = group of activities; **Activity** = a step; **Control flow** = branching/looping/dependencies. *(Dataset, Pipeline, Activity, Control flow were left blank in the notes - definitions added here.)* Equivalents: Linked service -> connection/external location/secret scope; Pipeline -> Lakeflow Job; Activity -> task; Control flow -> task dependencies, `if/else`, `for each` tasks.

### 13.2 Fivetran **[NOTES p40 rerun]**
Managed ELT. Principles: connectors as the base, predictable defaults, connector maintenance (support escalations, auto-retry of failed syncs), security (TLS + SSH against man-in-the-middle), fast data movement (incremental sync, large tables split into parallel threads). **Connect Card** = embeddable pop-up to collect credentials from your end users. **Connector SDK** (`pip install fivetran-connector-sdk`) runs your own Python connector in Fivetran's cloud (no connector yet, private API, unsupported formats, filtering/anonymizing sensitive data). Volumes: **extract** (read from source), **process** (before de-duplication, includes Fivetran metadata), **load** (loaded to destination after normalization).

### 13.3 Lakeflow (2025-2026) **[UPDATE]**
- **Lakeflow Connect** - managed connectors (Salesforce, SQL Server, Workday, SharePoint, ...) with incremental ingestion.
- **Lakeflow Spark Declarative Pipelines** - section 7. **Lakeflow Jobs** - orchestration (ex-Workflows). **Lakeflow Designer** - visual authoring.

### 13.4 Job schedulers outside Databricks **[NOTES p29]** - Apache Airflow, Dagster, Prefect (all have Databricks operators/integrations).

## 14. Data products & data contracts **[NOTES p34 rerun]**

A **data product** applies product thinking to data assets. Properties: **discoverable** (catalog entry, documented schema), **addressable** (stable, versioned interface), **trustworthy** (SLAs for freshness/completeness/accuracy enforced by checks), **self-describing** (docs, lineage, contract), **interoperable** (standards, naming), **valuable on its own**, **secure** (access control, privacy).

Imperatives: *consumption-ready*, *kept up to date* (per SLA), *approved for use* (governed by contracts).

Hierarchy: **source-aligned** (data as-is, minimal transformation, cleansed) -> **derived** (processed from source-aligned) -> **consumer-aligned** (dashboards, reports).

A **data contract** aligns domains in a federated-governance model and states: description (name, sources, attributes), schema, usage policies (tags, PII), quality checks/metrics, security (who can use), SLAs (last update, execution time), responsibilities (owner, maintainer, escalation path). In Databricks: Unity Catalog tags + constraints + expectations + lineage, shared via Delta Sharing or the Marketplace.

In [ ]:
# Contract-as-code sketch: a tiny quality gate against a table, returning a report
contract = {
  "table": "orders",
  "schema": {"order_id": "int", "customer": "string", "amount": "double"},
  "checks": [("amount_positive", "amount > 0"), ("customer_present", "customer IS NOT NULL")],
}
tbl = spark.table(contract["table"])
actual = {f.name: f.dataType.simpleString() for f in tbl.schema.fields}
schema_ok = all(actual.get(k) == v for k, v in contract["schema"].items())
rows = []
for name, cond in contract["checks"]:
    bad = tbl.filter(f"NOT ({cond})").count()
    rows.append((name, bad, bad == 0))
print("schema matches contract:", schema_ok)
display(spark.createDataFrame(rows, "check string, violations long, passed boolean"))

## 14b. Security & operations best practices **[NOTES U6-U11]**
| Topic | Notes | Current guidance **[UPDATE]** |
|---|---|---|
| Network | VNet injection (workspace in your VNet, dedicated subnets per workspace), VNet peering, Private Link, service endpoints, hub-and-spoke | Same; plus serverless network policies and NCC (network connectivity config) for serverless egress |
| Secrets | always in Key Vault | Databricks secret scopes backed by Key Vault, or UC service credentials; never in notebooks |
| Storage | do not put production data in the default DBFS root; deleting the workspace deletes it | DBFS root/mounts deprecated: use UC volumes / external locations |
| Identity | Entra ID, SCIM provisioning, conditional access, ADLS credential passthrough | Passthrough is legacy; Unity Catalog + managed identities; SCIM/automatic identity management |
| Audit | Azure Monitor audit logs (accounts, clusters, DBFS, jobs, ACLs, SSH, tables...) | UC `system.access.audit` system tables + diagnostic settings |
| HA/DR | two paired regions, Traffic Manager, workspaces via ARM templates | Same principles; Terraform/DABs; Delta Deep Clone / replication |
| Environments | one workspace per environment, cluster-scoped init scripts over global, cluster log delivery, cluster tags for chargeback, cluster ACLs | Same; compute policies, budgets, serverless usage policies, system billing tables |
| Libraries | notebook-scoped (`%pip`) vs cluster-scoped | Same; prefer wheels in volumes and serverless environments |
| Dev tooling | DB-Connect, Azure DevOps sync, CLI | **Databricks Connect** (Spark Connect), Git folders, DABs |
| Runtime tips | tune shuffle, partition, Delta Lake, ML runtime, Delta cache, Databricks Advisor | AQE, liquid clustering, Photon, predictive optimization; Advisor/HorovodRunner/ABS-AQS are obsolete |

## 14c. Spark ecosystem and miscellany **[NOTES p3, 13, 14, 21, 22, U6]**
- **Spark components:** Spark Core (scheduling, memory, fault recovery, storage access), Spark SQL, Structured Streaming, MLlib, GraphX (maintenance mode; GraphFrames). Written in Scala; APIs for Python, Scala, SQL, R, Java.
- **Cluster managers:** Standalone, Mesos (retired), Hadoop YARN, Kubernetes. `SparkSession` unifies SparkContext/SQLContext/HiveContext; `py4j` bridges Python to the JVM. **Spark Connect** (client-server) is the default on serverless/Databricks Connect.
- **Spark UI:** application UI per SparkSession, "master UI" per cluster; `spark.sparkContext.applicationId`.
- **Machine learning on Databricks:** AutoML, Spark ML, scikit-learn, PyTorch, TensorFlow, **MLflow** (tracking and model registry in Unity Catalog).
- **Microsoft Fabric comparison [NOTES p3]:** SaaS, all data in OneLake (ADLS-based, Delta/Parquet/CSV/JSON); *shortcuts* reference data without copying; Lakehouse = lake storage + warehouse querying; managed tables in Fabric's `Tables` area are Delta; the SQL analytics endpoint is read-only on Delta tables; **Direct Lake** semantic models read Delta files directly (a cache keeps hot data fresh); one Spark pool per workspace; permissions at workspace/item level. Databricks and Fabric can share data through OneLake shortcuts / UC mirroring.
- **AKS/Dv3 paragraph:** Databricks control/data plane services run as containers on Azure Kubernetes Service; the notes mention Dv3 VMs with NVMe SSDs and ~100 us I/O latency - hardware generations have since moved on (Dasv5/Dadsv6 etc.), the architecture statement is what matters.
- **Databricks vs open-source Spark [NOTES U6]:** workspace (collaboration + production jobs), Databricks Runtime (Photon, optimized I/O), serverless, enterprise security.
- **Resource limits [NOTES U10]:** the numbers in the notes (1000 jobs/hour, 150 concurrent jobs, 1500 API calls/hour, Azure subscription quotas) are outdated - check the current limits page before designing.

In [ ]:
# Handy introspection of the current session
display(spark.sql("SELECT current_version() AS dbsql_version, current_timezone() AS tz"))
display(spark.sql("SHOW TABLES"))

## 15. What changed since the notes (cheat sheet) **[UPDATE]**
| Old term / practice (notes) | Use now |
|---|---|
| Delta Live Tables, `dlt`, `CREATE LIVE TABLE` | Lakeflow Spark Declarative Pipelines, `pyspark.pipelines` (`dp`), `CREATE STREAMING TABLE` / `MATERIALIZED VIEW` |
| `APPLY CHANGES INTO` | AUTO CDC (same capability) |
| Databricks Workflows | Lakeflow Jobs |
| Databricks Repos | Git folders |
| DBFS root, `dbutils.fs.mount`, `dbfs:/` | UC volumes (`/Volumes/...`), external locations |
| Hive metastore (`hive_metastore.default`) | Unity Catalog (`catalog.schema.table`) |
| Z-ORDER + partitioning | Liquid clustering (`CLUSTER BY`), predictive optimization |
| High-concurrency / standard cluster modes | Standard (shared) / dedicated access modes; serverless |
| Azure AD | Microsoft Entra ID |
| ADLS credential passthrough | Unity Catalog credentials |
| `ignoreChanges` / `ignoreDeletes` | `skipChangeCommits` |
| `wasbs://`, `adl://` | `abfss://` |
| Spark 4.0 features list | Spark 4.1 / Databricks Runtime 18 (GA 2026-06); `QUALIFY`, SQL scripting, stored procedures, VARIANT, pipe syntax, parameter markers |
| Stored procedures / loops "don't exist" | SQL scripting and `CREATE PROCEDURE` exist |
| `pandas-profiling` | `ydata-profiling` |
| `sumDistinct` | `sum_distinct` |
| `spark.sql.execution.arrow.enabled` | `spark.sql.execution.arrow.pyspark.enabled` (on by default) |
| Fabric "default dataset" | default semantic model |
| Zookeeper-based Kafka | KRaft only (Kafka 4.0+) |
| Python-script CI/CD against REST API | Databricks Asset Bundles + OIDC |

*End of notebook. Verify product names, versions and limits against the official documentation - they change frequently.*